# RetinaFace Detection Test (Random Testset)

This notebook evaluates the `retina-face` library on a curated random testset from this workspace. It installs dependencies (global env), loads images from `testsets/random_testset.json`, runs face detection, visualizes results, and benchmarks performance.



Steps:

1. Install and import dependencies

2. Configure paths and output directory

3. Load and sample the random testset

4. Run RetinaFace detection and visualize

5. Benchmark detection time and save annotated outputs

In [ ]:
# Imports & environment check

import os

import json

import time

import random

from datetime import datetime



import numpy as np

import matplotlib.pyplot as plt

from PIL import Image

import cv2



try:

    from retinaface import RetinaFace

    RETINAFACE_AVAILABLE = True

except Exception as e:

    RETINAFACE_AVAILABLE = False

    print("RetinaFace import failed:", e)



print("Env OK:")

print({

    "python": os.sys.version.split()[0],

    "numpy": np.__version__,

    "opencv": cv2.__version__,

    "matplotlib": plt.matplotlib.__version__,

    "pillow": Image.__version__,

    "retinaface": RETINAFACE_AVAILABLE

})

In [ ]:
# Config: paths and output dir

BASE_DIR = "/app"  # workspace root

TESTSET_PATH = os.path.join(BASE_DIR, "testsets", "random_testset.json")

SAMPLE_SIZE = 20  # adjust as desired



timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIR = os.path.join(BASE_DIR, "image_outputs", f"{timestamp}_retina_face_random_test")

os.makedirs(OUTPUT_DIR, exist_ok=True)



print("Config:")

print({

    "TESTSET_PATH": TESTSET_PATH,

    "SAMPLE_SIZE": SAMPLE_SIZE,

    "OUTPUT_DIR": OUTPUT_DIR

})

In [ ]:
# Load testset and sample image paths

def load_random_testset(testset_path: str):

    with open(testset_path, "r") as f:

        entries = json.load(f)

    # normalize to absolute paths

    for e in entries:

        rel_path = e.get("path")

        e["abs_path"] = os.path.join(BASE_DIR, rel_path)

    return entries



def sample_entries(entries, n: int):

    valid = [e for e in entries if os.path.exists(e["abs_path"])]

    if len(valid) == 0:

        print("No valid image paths found. Check TESTSET_PATH and image locations.")

        return []

    if n >= len(valid):

        return valid

    return random.sample(valid, n)



entries = load_random_testset(TESTSET_PATH)

sampled = sample_entries(entries, SAMPLE_SIZE)

print(f"Loaded {len(entries)} entries; sampled {len(sampled)} valid images.")

print("Example:", sampled[0]["abs_path"] if sampled else None)

In [ ]:
# Detection and visualization helpers

def detect_faces_retina(image_path: str):

    if not RETINAFACE_AVAILABLE:

        raise RuntimeError("RetinaFace library not available. Please ensure installation.")

    try:

        faces = RetinaFace.detect_faces(img_path=image_path)

        return faces  # dict keyed by face id; each has 'facial_area' and landmarks

    except Exception as e:

        print(f"Detection error on {image_path}: {e}")

        return None



def annotate_faces_bgr(image_bgr: np.ndarray, faces_dict):

    if faces_dict in (None, False):

        return image_bgr

    annotated = image_bgr.copy()

    # faces_dict is a dict like {"face_1": {"facial_area": [x1,y1,x2,y2], ...}, ...}

    for k, v in faces_dict.items():

        box = v.get("facial_area")

        if box and len(box) == 4:

            x1, y1, x2, y2 = box

            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)

            cv2.putText(annotated, k, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1, cv2.LINE_AA)

    return annotated



def show_image_bgr(image_bgr: np.ndarray, title: str = None):

    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 6))

    plt.imshow(rgb)

    plt.axis('off')

    if title:

        plt.title(title)

    plt.show()



# Visualize one sample image

if sampled:

    sample_path = sampled[0]["abs_path"]

    img_bgr = cv2.imread(sample_path)

    faces = detect_faces_retina(sample_path)

    annotated = annotate_faces_bgr(img_bgr, faces)

    face_count = 0 if (faces in (None, False)) else len(faces)

    show_image_bgr(annotated, title=f"{os.path.basename(sample_path)} - faces: {face_count}")

else:

    print("No sampled images to visualize.")

In [ ]:
# Benchmark across sampled images and save outputs

results = []

start_all = time.time()



for e in sampled:

    path = e["abs_path"]

    img_name = os.path.basename(path)

    img_bgr = cv2.imread(path)

    if img_bgr is None:

        print("Failed to read:", path)

        results.append({"path": path, "faces": 0, "error": "read-failed", "time_s": None})

        continue

    t0 = time.time()

    faces = detect_faces_retina(path)

    t1 = time.time()

    dt = t1 - t0

    count = 0 if (faces in (None, False)) else len(faces)

    annotated = annotate_faces_bgr(img_bgr, faces)

    out_path = os.path.join(OUTPUT_DIR, f"annotated_{img_name}")

    cv2.imwrite(out_path, annotated)

    results.append({"path": path, "faces": count, "time_s": dt, "out": out_path})



end_all = time.time()

total_time = end_all - start_all



if results:

    times = [r["time_s"] for r in results if r["time_s"] is not None]

    faces_counts = [r["faces"] for r in results if r["time_s"] is not None]

    avg_time = sum(times)/len(times) if times else None

    avg_faces = sum(faces_counts)/len(faces_counts) if faces_counts else None

    print("Benchmark summary:")

    print({

        "images_processed": len(results),

        "total_time_s": round(total_time, 3),

        "avg_time_per_image_s": None if avg_time is None else round(avg_time, 3),

        "avg_faces_per_image": None if avg_faces is None else round(avg_faces, 3),

        "output_dir": OUTPUT_DIR

    })

else:

    print("No results to summarize.")



# Preview a few outputs

preview = results[:4]

for r in preview:

    p = r.get("out")

    if p and os.path.exists(p):

        img_bgr = cv2.imread(p)

        show_image_bgr(img_bgr, title=f"{os.path.basename(p)} | faces: {r['faces']} | {round(r['time_s'],3)}s")